# 🔐 Quantum-Inspired Cybersecurity: Network Intrusion Detection
### Using Quantum Walk + Quantum Fourier Transform (QFT) on NSL-KDD Dataset

---

## 📌 Project Overview
This notebook implements the **same algorithms** used in the backend project:
- **Quantum Walk** → builds a network graph, detects anomalous IPs by how probability distributes
- **QFT (via DFT)** → analyzes time-series traffic patterns, detects beaconing/periodic attacks
- **Threat Scoring** → combines both scores to classify: `Normal`, `Suspicious`, `Attack`

## 📂 Dataset
**NSL-KDD** from Kaggle:  
https://www.kaggle.com/datasets/sampadab17/network-intrusion-detection

Files used:
- `Train_data.csv` → training
- `Test_data.csv`  → testing

## 🔬 Algorithm Flow
```
Raw CSV → Feature Engineering → Graph Builder
                                      │
                         ┌────────────┴────────────┐
                   Quantum Walk               QFT Analysis
                   (node anomaly)         (traffic periodicity)
                         └────────────┬────────────┘
                               Threat Scoring
                         (0.6 × QW + 0.4 × QFT)
                                      │
                         Normal / Suspicious / Attack
                                      │
                          Evaluation + Visualizations
```

## 📦 Step 1 — Install & Import Libraries

In [ ]:
# Install required libraries
!pip install qiskit qiskit-aer numpy pandas matplotlib seaborn scikit-learn networkx -q
print('✅ All libraries installed')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import warnings
import random
from datetime import datetime, timedelta
from collections import defaultdict
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.preprocessing import LabelEncoder

# Qiskit imports
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
plt.style.use('seaborn-v0_8-darkgrid')

print('✅ All imports successful')

## 📂 Step 2 — Load Dataset

In [ ]:
# ── Upload files from Kaggle ──────────────────────────────────────────────────
# Option A: Upload manually
from google.colab import files
print('📤 Please upload Train_data.csv and Test_data.csv from Kaggle')
uploaded = files.upload()

In [ ]:
# ── Load Train and Test CSVs ──────────────────────────────────────────────────
train_df = pd.read_csv('Train_data.csv')
test_df  = pd.read_csv('Test_data.csv')

# Strip whitespace from columns
train_df.columns = train_df.columns.str.strip()
test_df.columns  = test_df.columns.str.strip()

print(f'✅ Train set: {train_df.shape[0]:,} rows × {train_df.shape[1]} columns')
print(f'✅ Test set:  {test_df.shape[0]:,} rows × {test_df.shape[1]} columns')
print(f'\n📋 Columns: {train_df.columns.tolist()}')
train_df.head(3)

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
label_col = 'class' if 'class' in train_df.columns else 'attack_type'

print('🔍 TRAIN — Attack type distribution:')
train_counts = train_df[label_col].value_counts()
print(train_counts.to_string())

print('\n🔍 TEST — Attack type distribution:')
test_counts = test_df[label_col].value_counts()
print(test_counts.to_string())

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
train_counts.head(10).plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Train Set — Top 10 Attack Types', fontweight='bold')
axes[0].set_xlabel('Attack Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

test_counts.head(10).plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Test Set — Top 10 Attack Types', fontweight='bold')
axes[1].set_xlabel('Attack Type')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Chart saved: 01_class_distribution.png')

## ⚙️ Step 3 — Feature Engineering & IP Generation

In [ ]:
# ── Attack group mapping (same as backend project) ────────────────────────────
DOS_ATTACKS   = {'back','land','neptune','pod','smurf','teardrop',
                 'apache2','udpstorm','processtable','worm'}
PROBE_ATTACKS = {'ipsweep','nmap','portsweep','satan','mscan','saint'}
R2L_ATTACKS   = {'ftp_write','guess_passwd','imap','multihop','phf','spy',
                 'warezclient','warezmaster','sendmail','named',
                 'snmpgetattack','snmpguess','xlock','xsnoop','httptunnel'}
U2R_ATTACKS   = {'buffer_overflow','loadmodule','perl','rootkit',
                 'ps','sqlattack','xterm'}

def get_attack_group(label):
    label = str(label).strip().lower()
    if label == 'normal': return 'normal'
    if label in DOS_ATTACKS:   return 'dos'
    if label in PROBE_ATTACKS: return 'probe'
    if label in R2L_ATTACKS:   return 'r2l'
    if label in U2R_ATTACKS:   return 'u2r'
    return 'dos'  # unknown → treat as dos

# ── IP pools ──────────────────────────────────────────────────────────────────
ATTACKER_IPS = [f'192.168.99.{i}' for i in range(1, 16)]
NORMAL_IPS   = [f'192.168.1.{i}'  for i in range(1, 60)]
SERVER_IPS   = [f'10.0.0.{i}'     for i in range(1, 30)]

def assign_ips(label):
    group = get_attack_group(label)
    if group == 'normal':
        return random.choice(NORMAL_IPS), random.choice(SERVER_IPS)
    elif group == 'dos':
        return random.choice(ATTACKER_IPS[:5]), random.choice(SERVER_IPS)
    elif group == 'probe':
        return random.choice(ATTACKER_IPS[5:8]), random.choice(SERVER_IPS)
    elif group in ('r2l', 'u2r'):
        return random.choice(ATTACKER_IPS[8:12]), SERVER_IPS[0]
    return random.choice(ATTACKER_IPS), random.choice(SERVER_IPS)

def generate_timestamp(label, base_time, index):
    group = get_attack_group(label)
    if group == 'normal':
        return base_time + timedelta(seconds=random.randint(0, 28800))
    elif group == 'dos':
        return base_time + timedelta(seconds=random.randint(0, 300),
                                     milliseconds=random.randint(0, 999))
    elif group == 'probe':
        return base_time + timedelta(seconds=random.randint(0, 600))
    else:
        return base_time + timedelta(minutes=random.randint(0, 120))

PROTOCOL_MAP = {'tcp': 'TCP', 'udp': 'UDP', 'icmp': 'ICMP'}

def engineer_features(df, sample_size=2000, label_col='class'):
    """Convert NSL-KDD raw data into network traffic format."""
    # Sample: 60% attack, 40% normal
    attack_df = df[df[label_col].str.strip().str.lower() != 'normal']
    normal_df = df[df[label_col].str.strip().str.lower() == 'normal']

    n_attack = min(len(attack_df), int(sample_size * 0.6))
    n_normal = min(len(normal_df), sample_size - n_attack)

    sampled = pd.concat([
        attack_df.sample(n=n_attack, random_state=42),
        normal_df.sample(n=n_normal, random_state=42)
    ]).sample(frac=1, random_state=42).reset_index(drop=True)

    base_time = datetime(2024, 1, 15, 8, 0, 0)
    labels    = sampled[label_col].tolist()

    src_ips, dst_ips, timestamps = [], [], []
    for i, label in enumerate(labels):
        src, dst = assign_ips(label)
        src_ips.append(src)
        dst_ips.append(dst)
        timestamps.append(generate_timestamp(label, base_time, i))

    packet_sizes = pd.to_numeric(sampled['src_bytes'], errors='coerce') \
                     .fillna(64).abs().clip(64, 65535).astype(int)
    protocols    = sampled['protocol_type'].str.strip().str.lower() \
                     .map(PROTOCOL_MAP).fillna('OTHER')

    result = pd.DataFrame({
        'source_ip':    src_ips,
        'dest_ip':      dst_ips,
        'protocol':     protocols.values,
        'packet_size':  packet_sizes.values,
        'timestamp':    timestamps,
        'attack_type':  labels,
        'attack_group': [get_attack_group(l) for l in labels],
        'true_label':   ['normal' if str(l).strip().lower()=='normal' else 'attack'
                         for l in labels]
    })
    return result

print('⚙️  Engineering features...')
train_net = engineer_features(train_df, sample_size=2000)
test_net  = engineer_features(test_df,  sample_size=500)

print(f'✅ Train network data: {len(train_net):,} rows')
print(f'✅ Test network data:  {len(test_net):,} rows')
print(f'\nAttack group distribution (train):')
print(train_net['attack_group'].value_counts())
train_net.head()

## 🕸️ Step 4 — Network Graph Builder

In [ ]:
class NetworkGraphBuilder:
    """
    Builds a weighted undirected graph from network traffic logs.
    Nodes = unique IP addresses
    Edges = communications between IPs (weight = number of packets)
    """
    def __init__(self):
        self.graph      = nx.Graph()
        self.node_info  = {}   # ip → {total_connections, packet_volume}
        self.edge_weights = defaultdict(int)

    def build(self, df):
        """Build graph from dataframe with source_ip, dest_ip, packet_size."""
        for _, row in df.iterrows():
            src  = row['source_ip']
            dst  = row['dest_ip']
            size = row['packet_size']

            # Add nodes
            for ip in [src, dst]:
                if ip not in self.node_info:
                    self.node_info[ip] = {'total_connections': 0, 'packet_volume': 0}
                self.node_info[ip]['total_connections'] += 1
                self.node_info[ip]['packet_volume']     += size

            # Add/update edge
            key = tuple(sorted([src, dst]))
            self.edge_weights[key] += 1

        # Build NetworkX graph
        self.graph.clear()
        for ip, info in self.node_info.items():
            self.graph.add_node(ip, **info)
        for (src, dst), weight in self.edge_weights.items():
            self.graph.add_edge(src, dst, weight=weight)

        print(f'🕸️  Graph built: {self.graph.number_of_nodes()} nodes, '
              f'{self.graph.number_of_edges()} edges')
        return self.graph

    def get_adjacency(self):
        """Returns dict: ip → [(neighbour_ip, weight), ...]"""
        adj = {}
        for node in self.graph.nodes():
            adj[node] = [(nbr, self.graph[node][nbr]['weight'])
                         for nbr in self.graph.neighbors(node)]
        return adj

    def visualize(self, title='Network Graph', max_nodes=50):
        """Visualize the network graph with attack/normal coloring."""
        G = self.graph
        if G.number_of_nodes() > max_nodes:
            # Show only top connected nodes
            top_nodes = sorted(G.degree, key=lambda x: x[1], reverse=True)[:max_nodes]
            G = G.subgraph([n for n, _ in top_nodes])

        plt.figure(figsize=(14, 8))
        pos = nx.spring_layout(G, seed=42, k=2)

        # Color nodes: attacker IPs red, server IPs blue, normal green
        node_colors = []
        for node in G.nodes():
            if '192.168.99' in node:  node_colors.append('#FF4444')
            elif '10.0.0'   in node:  node_colors.append('#4488FF')
            else:                      node_colors.append('#44BB44')

        edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
        max_w = max(edge_weights) if edge_weights else 1
        edge_widths = [1 + 4 * (w / max_w) for w in edge_weights]

        nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                               node_size=300, alpha=0.9)
        nx.draw_networkx_edges(G, pos, width=edge_widths,
                               alpha=0.5, edge_color='gray')
        nx.draw_networkx_labels(G, pos, font_size=6)

        legend = [
            mpatches.Patch(color='#FF4444', label='Attacker IPs (192.168.99.x)'),
            mpatches.Patch(color='#4488FF', label='Server IPs (10.0.0.x)'),
            mpatches.Patch(color='#44BB44', label='Normal User IPs (192.168.1.x)'),
        ]
        plt.legend(handles=legend, loc='upper left')
        plt.title(title, fontsize=14, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        plt.savefig('02_network_graph.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('📊 Chart saved: 02_network_graph.png')

# Build graph from training data
print('Building network graph from training data...')
graph_builder = NetworkGraphBuilder()
G_train = graph_builder.build(train_net)
graph_builder.visualize('Network Traffic Graph — Training Data')

## ⚛️ Step 5 — Quantum Walk Algorithm

In [ ]:
class QuantumWalkAnalyzer:
    """
    Discrete-time Quantum Walk on the network graph.

    Same algorithm as the C# backend QuantumWalkHelper:
    1. Initialize equal amplitudes: 1/√N for each node
    2. Spread amplitude to neighbors weighted by edge weight
    3. Normalize state vector at each step
    4. After T steps: probability = amplitude²
    5. Anomaly score = how far node deviates from mean probability

    High anomaly score → node is a hub of attack traffic.
    """

    def __init__(self, steps=8):
        self.steps   = steps
        self.results = {}   # ip → {probability, anomaly_score}

    def run(self, adjacency):
        """
        adjacency: dict of ip → [(neighbour, weight), ...]
        Returns: dict of ip → anomaly_score (0 to 1)
        """
        ips = list(adjacency.keys())
        n   = len(ips)
        if n == 0:
            return {}

        ip_to_idx = {ip: i for i, ip in enumerate(ips)}

        # ── Step 1: Initialize uniform superposition ──────────────────────────
        amplitudes = np.ones(n) / np.sqrt(n)

        # ── Step 2: Walk (T steps) ────────────────────────────────────────────
        for step in range(self.steps):
            next_amplitudes = np.zeros(n)

            for i, ip in enumerate(ips):
                neighbours = adjacency[ip]

                if not neighbours:
                    # Isolated node — keeps its own amplitude
                    next_amplitudes[i] += amplitudes[i]
                    continue

                total_weight = sum(w for _, w in neighbours)
                for (neighbour, weight) in neighbours:
                    j = ip_to_idx[neighbour]
                    transfer = weight / total_weight
                    next_amplitudes[j] += amplitudes[i] * transfer

            # Normalize state vector (preserve unit norm)
            norm = np.linalg.norm(next_amplitudes)
            if norm > 0:
                next_amplitudes /= norm

            amplitudes = next_amplitudes

        # ── Step 3: Compute probabilities ─────────────────────────────────────
        probabilities = amplitudes ** 2

        # ── Step 4: Compute anomaly scores ────────────────────────────────────
        mean_prob = np.mean(probabilities)
        std_prob  = np.std(probabilities)

        # Normalize anomaly to [0, 1]
        max_prob  = np.max(probabilities)

        self.results = {}
        for i, ip in enumerate(ips):
            prob    = probabilities[i]
            anomaly = abs(prob - mean_prob) / (std_prob + 1e-10)
            # Normalize to [0, 1] relative to max deviation in this run
            max_anomaly = max(abs(probabilities - mean_prob)) / (std_prob + 1e-10)
            anomaly_norm = anomaly / (max_anomaly + 1e-10)

            self.results[ip] = {
                'probability':   round(float(prob), 6),
                'anomaly_score': round(float(anomaly_norm), 4)
            }

        return {ip: v['anomaly_score'] for ip, v in self.results.items()}

    def get_top_anomalies(self, n=10):
        """Return top N most anomalous IPs."""
        return sorted(
            self.results.items(),
            key=lambda x: x[1]['anomaly_score'],
            reverse=True
        )[:n]

    def plot_scores(self, title='Quantum Walk — Anomaly Scores per IP'):
        if not self.results:
            print('No results yet. Run .run() first.')
            return

        ips    = list(self.results.keys())
        scores = [self.results[ip]['anomaly_score'] for ip in ips]

        # Color by IP group
        colors = ['#FF4444' if '99' in ip else
                  '#4488FF' if '10.0' in ip else '#44BB44'
                  for ip in ips]

        sorted_pairs = sorted(zip(scores, ips, colors), reverse=True)
        scores_s, ips_s, colors_s = zip(*sorted_pairs[:30])  # top 30

        plt.figure(figsize=(14, 6))
        bars = plt.bar(range(len(ips_s)), scores_s, color=colors_s, edgecolor='black', alpha=0.85)
        plt.xticks(range(len(ips_s)), ips_s, rotation=90, fontsize=8)
        plt.axhline(y=0.65, color='orange', linestyle='--', linewidth=2, label='Suspicious threshold (0.65)')
        plt.axhline(y=0.85, color='red',    linestyle='--', linewidth=2, label='Attack threshold (0.85)')
        plt.xlabel('IP Address')
        plt.ylabel('Anomaly Score')
        plt.title(title, fontweight='bold')
        legend = [
            mpatches.Patch(color='#FF4444', label='Attacker IPs'),
            mpatches.Patch(color='#4488FF', label='Server IPs'),
            mpatches.Patch(color='#44BB44', label='Normal IPs'),
        ]
        plt.legend(handles=legend + [
            plt.Line2D([0],[0], color='orange', linestyle='--', label='Suspicious (0.65)'),
            plt.Line2D([0],[0], color='red',    linestyle='--', label='Attack (0.85)'),
        ])
        plt.tight_layout()
        plt.savefig('03_quantum_walk_scores.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('📊 Chart saved: 03_quantum_walk_scores.png')


# ── Run Quantum Walk on Training Data ────────────────────────────────────────
print('⚛️  Running Quantum Walk on training graph...')
qw_analyzer = QuantumWalkAnalyzer(steps=8)
qw_scores   = qw_analyzer.run(graph_builder.get_adjacency())

print(f'\n✅ Quantum Walk complete for {len(qw_scores)} nodes')
print('\n🔴 Top 10 most anomalous nodes:')
for ip, info in qw_analyzer.get_top_anomalies(10):
    print(f'   {ip:20s}  anomaly={info["anomaly_score"]:.4f}  prob={info["probability"]:.6f}')

qw_analyzer.plot_scores('Quantum Walk — Anomaly Scores (Training Data)')

## 🌊 Step 6 — QFT (Quantum Fourier Transform) via DFT

In [ ]:
class QFTAnalyzer:
    """
    Quantum Fourier Transform simulation using Discrete Fourier Transform.

    Same algorithm as the C# backend QftAnalysisHelper:
    1. Bin each IP's traffic into N time buckets
    2. Apply DFT to get frequency spectrum
    3. Find dominant frequency (skip DC component k=0)
    4. Periodicity score = peak_magnitude / total_magnitude

    High periodicity → machine-like regular traffic → beaconing / attack.

    Also builds a Qiskit circuit to demonstrate QFT on qubit register.
    """

    def __init__(self, bucket_count=16):
        self.bucket_count = bucket_count   # must be power of 2
        self.results      = {}             # ip → {dominant_freq, periodicity_score}

    # ── Classical DFT (same as backend) ──────────────────────────────────────
    def _compute_dft(self, signal):
        n          = len(signal)
        magnitudes = np.zeros(n // 2)

        for k in range(n // 2):
            real = sum(signal[t] * np.cos(2 * np.pi * k * t / n) for t in range(n))
            imag = sum(signal[t] * np.sin(2 * np.pi * k * t / n) for t in range(n))
            magnitudes[k] = np.sqrt(real**2 + imag**2)

        # Skip DC component (k=0), find dominant frequency
        dominant_k = np.argmax(magnitudes[1:]) + 1
        total_mag  = magnitudes[1:].sum()
        peak_mag   = magnitudes[dominant_k]

        periodicity = float(peak_mag / total_mag) if total_mag > 0 else 0.0
        return int(dominant_k), min(periodicity, 1.0)

    # ── Qiskit QFT Circuit (for demonstration) ────────────────────────────────
    def build_qft_circuit(self, n_qubits=4):
        """
        Builds a proper QFT circuit using Qiskit.
        n_qubits corresponds to log2(bucket_count).
        """
        qr  = QuantumRegister(n_qubits, 'q')
        cr  = ClassicalRegister(n_qubits, 'c')
        qc  = QuantumCircuit(qr, cr)

        # Initialize to uniform superposition
        qc.h(qr)

        # QFT circuit
        for j in range(n_qubits):
            qc.h(qr[j])
            for k in range(j + 1, n_qubits):
                angle = np.pi / (2 ** (k - j))
                qc.cp(angle, qr[k], qr[j])

        # Swap qubits (bit-reversal permutation)
        for i in range(n_qubits // 2):
            qc.swap(qr[i], qr[n_qubits - i - 1])

        qc.measure(qr, cr)
        return qc

    def run_qft_circuit(self, n_qubits=4, shots=1024):
        """Run QFT circuit on Qiskit simulator and return measurement counts."""
        qc      = self.build_qft_circuit(n_qubits)
        backend = AerSimulator()
        job     = backend.run(qc, shots=shots)
        counts  = job.result().get_counts()
        return qc, counts

    # ── Main analysis ─────────────────────────────────────────────────────────
    def run(self, df):
        """
        df: DataFrame with source_ip, dest_ip, packet_size, timestamp
        Returns: dict of ip → periodicity_score (0 to 1)
        """
        all_ips = set(df['source_ip'].tolist() + df['dest_ip'].tolist())

        # Get time range for bucketing
        min_ts = df['timestamp'].min()
        max_ts = df['timestamp'].max()
        span   = (max_ts - min_ts).total_seconds()
        if span <= 0: span = 1

        self.results = {}

        for ip in all_ips:
            # Get all traffic involving this IP
            ip_df = df[(df['source_ip'] == ip) | (df['dest_ip'] == ip)]
            if len(ip_df) == 0:
                continue

            # Build time-series signal (buckets)
            signal = np.zeros(self.bucket_count)
            for _, row in ip_df.iterrows():
                rel_pos = (row['timestamp'] - min_ts).total_seconds() / span
                bucket  = min(int(rel_pos * self.bucket_count), self.bucket_count - 1)
                signal[bucket] += row['packet_size']

            # DFT → periodicity score
            dominant_freq, periodicity = self._compute_dft(signal)

            self.results[ip] = {
                'dominant_frequency': dominant_freq,
                'periodicity_score':  round(periodicity, 4),
                'signal':             signal
            }

        return {ip: v['periodicity_score'] for ip, v in self.results.items()}

    def plot_qft_circuit(self):
        """Show the Qiskit QFT circuit diagram."""
        qc = self.build_qft_circuit(n_qubits=4)
        print('⚛️  QFT Quantum Circuit (4 qubits = 16 frequency bins):')
        print(qc.draw(output='text'))

    def plot_periodicity_scores(self, title='QFT — Periodicity Scores per IP'):
        if not self.results:
            print('No results. Run .run() first.')
            return

        ips    = list(self.results.keys())
        scores = [self.results[ip]['periodicity_score'] for ip in ips]
        colors = ['#FF4444' if '99' in ip else
                  '#4488FF' if '10.0' in ip else '#44BB44'
                  for ip in ips]

        sorted_pairs = sorted(zip(scores, ips, colors), reverse=True)
        scores_s, ips_s, colors_s = zip(*sorted_pairs[:30])

        plt.figure(figsize=(14, 6))
        plt.bar(range(len(ips_s)), scores_s, color=colors_s, edgecolor='black', alpha=0.85)
        plt.xticks(range(len(ips_s)), ips_s, rotation=90, fontsize=8)
        plt.axhline(y=0.5, color='orange', linestyle='--', linewidth=2, label='Beaconing threshold')
        plt.xlabel('IP Address')
        plt.ylabel('Periodicity Score')
        plt.title(title, fontweight='bold')
        legend = [
            mpatches.Patch(color='#FF4444', label='Attacker IPs'),
            mpatches.Patch(color='#4488FF', label='Server IPs'),
            mpatches.Patch(color='#44BB44', label='Normal IPs'),
            plt.Line2D([0],[0], color='orange', linestyle='--', label='Beaconing threshold'),
        ]
        plt.legend(handles=legend)
        plt.tight_layout()
        plt.savefig('04_qft_periodicity.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('📊 Chart saved: 04_qft_periodicity.png')

    def plot_traffic_signal(self, ip):
        """Plot time-series signal for a specific IP."""
        if ip not in self.results:
            print(f'IP {ip} not found in results.')
            return
        signal = self.results[ip]['signal']
        score  = self.results[ip]['periodicity_score']
        freq   = self.results[ip]['dominant_frequency']

        freqs = np.fft.rfftfreq(len(signal))
        fft_m = np.abs(np.fft.rfft(signal))

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        axes[0].plot(signal, marker='o', color='steelblue', linewidth=2)
        axes[0].set_title(f'{ip} — Time-Series Traffic Signal')
        axes[0].set_xlabel('Time Bucket')
        axes[0].set_ylabel('Total Packet Size')
        axes[0].fill_between(range(len(signal)), signal, alpha=0.3)

        axes[1].stem(freqs, fft_m, linefmt='C1-', markerfmt='C1o', basefmt='gray')
        axes[1].set_title(f'Frequency Spectrum — Periodicity: {score:.3f}  DomFreq: {freq}')
        axes[1].set_xlabel('Frequency')
        axes[1].set_ylabel('Magnitude')

        plt.suptitle(f'QFT Analysis: {ip}', fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'05_signal_{ip.replace(".","_")}.png', dpi=150, bbox_inches='tight')
        plt.show()


# ── Show QFT circuit ──────────────────────────────────────────────────────────
qft_analyzer = QFTAnalyzer(bucket_count=16)
qft_analyzer.plot_qft_circuit()

# ── Run QFT analysis ──────────────────────────────────────────────────────────
print('\n🌊 Running QFT analysis on training data...')
qft_scores = qft_analyzer.run(train_net)

print(f'\n✅ QFT complete for {len(qft_scores)} nodes')
print('\n🔴 Top 10 most periodic (beaconing) nodes:')
top_qft = sorted(qft_scores.items(), key=lambda x: x[1], reverse=True)[:10]
for ip, score in top_qft:
    print(f'   {ip:20s}  periodicity={score:.4f}')

qft_analyzer.plot_periodicity_scores('QFT — Traffic Periodicity Scores (Training Data)')

In [ ]:
# ── Plot signal for top attacker IP ──────────────────────────────────────────
top_attacker = sorted(qft_scores.items(), key=lambda x: x[1], reverse=True)[0][0]
print(f'Plotting signal for most periodic IP: {top_attacker}')
qft_analyzer.plot_traffic_signal(top_attacker)

# ── Run actual Qiskit QFT circuit ────────────────────────────────────────────
print('\n⚛️  Running Qiskit QFT circuit on simulator...')
qc, counts = qft_analyzer.run_qft_circuit(n_qubits=4, shots=1024)
print(f'QFT measurement counts (top 8): {dict(list(sorted(counts.items(), key=lambda x: x[1], reverse=True))[:8])}')

# Plot measurement distribution
plt.figure(figsize=(12, 4))
states  = list(counts.keys())
freq_c  = list(counts.values())
plt.bar(states, freq_c, color='purple', alpha=0.7, edgecolor='black')
plt.xticks(rotation=90, fontsize=8)
plt.xlabel('Quantum State')
plt.ylabel('Measurement Count')
plt.title('⚛️ Qiskit QFT Circuit — Measurement Distribution (1024 shots)', fontweight='bold')
plt.tight_layout()
plt.savefig('06_qft_circuit_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Chart saved: 06_qft_circuit_results.png')

## 🎯 Step 7 — Threat Scoring & Classification

In [ ]:
class ThreatScorer:
    """
    Combines Quantum Walk + QFT scores → threat classification.
    Same formula as C# ThreatScoringHelper:

    ThreatScore = (QW_anomaly × 0.6) + (QFT_periodicity × 0.4)

    Classification:
        score >= attack_threshold    → Attack
        score >= anomaly_threshold   → Suspicious
        otherwise                    → Normal
    """

    def __init__(self, qw_weight=0.6, qft_weight=0.4,
                 anomaly_threshold=0.35, attack_threshold=0.65):
        self.qw_weight         = qw_weight
        self.qft_weight        = qft_weight
        self.anomaly_threshold = anomaly_threshold
        self.attack_threshold  = attack_threshold
        self.results           = {}

    def score(self, qw_scores, qft_scores):
        """
        qw_scores:  dict ip → anomaly_score  (0–1)
        qft_scores: dict ip → periodicity_score (0–1)
        """
        all_ips = set(list(qw_scores.keys()) + list(qft_scores.keys()))

        self.results = {}
        for ip in all_ips:
            qw_s  = qw_scores.get(ip, 0)
            qft_s = qft_scores.get(ip, 0)

            threat_score = round(
                (self.qw_weight * qw_s) + (self.qft_weight * qft_s), 4
            )

            if threat_score >= self.attack_threshold:
                level = 'Attack'
            elif threat_score >= self.anomaly_threshold:
                level = 'Suspicious'
            else:
                level = 'Normal'

            self.results[ip] = {
                'qw_score':    round(qw_s,  4),
                'qft_score':   round(qft_s, 4),
                'threat_score': threat_score,
                'threat_level': level
            }

        return self.results

    def get_summary(self):
        from collections import Counter
        counts = Counter(v['threat_level'] for v in self.results.values())
        return dict(counts)

    def to_dataframe(self):
        rows = []
        for ip, v in self.results.items():
            rows.append({'ip': ip, **v})
        return pd.DataFrame(rows).sort_values('threat_score', ascending=False)


# ── Score training data ───────────────────────────────────────────────────────
print('🎯 Running Threat Scoring on training data...')
scorer       = ThreatScorer(qw_weight=0.6, qft_weight=0.4,
                             anomaly_threshold=0.35, attack_threshold=0.65)
train_scores = scorer.score(qw_scores, qft_scores)

summary = scorer.get_summary()
print(f'\n📊 Training Results Summary:')
for level in ['Attack', 'Suspicious', 'Normal']:
    count = summary.get(level, 0)
    emoji = '🔴' if level == 'Attack' else '🟡' if level == 'Suspicious' else '🟢'
    print(f'   {emoji} {level:12s}: {count} IPs')

results_df = scorer.to_dataframe()
print(f'\n🔴 Top 10 Threats:')
print(results_df[results_df['threat_level'].isin(['Attack','Suspicious'])]
      .head(10)[['ip','qw_score','qft_score','threat_score','threat_level']]
      .to_string(index=False))

## 📊 Step 8 — Visualizations

In [ ]:
# ── 1. Threat level pie chart ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Pie chart
summary = scorer.get_summary()
labels  = [k for k in ['Attack','Suspicious','Normal'] if k in summary]
sizes   = [summary[k] for k in labels]
colors  = ['#FF4444','#FFA500','#44BB44']
axes[0].pie(sizes, labels=labels, colors=colors[:len(labels)],
            autopct='%1.1f%%', startangle=90, shadow=True)
axes[0].set_title('Threat Level Distribution\n(Training IPs)', fontweight='bold')

# ── 2. Scatter: QW score vs QFT score ────────────────────────────────────────
color_map = {'Attack': '#FF4444', 'Suspicious': '#FFA500', 'Normal': '#44BB44'}
for level in ['Normal', 'Suspicious', 'Attack']:
    sub = results_df[results_df['threat_level'] == level]
    axes[1].scatter(sub['qw_score'], sub['qft_score'],
                    c=color_map[level], label=level, s=60, alpha=0.7, edgecolors='black')
axes[1].axvline(x=0.35, color='gray', linestyle=':', alpha=0.7)
axes[1].axhline(y=0.35, color='gray', linestyle=':', alpha=0.7)
axes[1].set_xlabel('Quantum Walk Anomaly Score')
axes[1].set_ylabel('QFT Periodicity Score')
axes[1].set_title('QW Score vs QFT Score\n(Color = Threat Level)', fontweight='bold')
axes[1].legend()

# ── 3. Threat score histogram ─────────────────────────────────────────────────
for level in ['Normal', 'Suspicious', 'Attack']:
    sub = results_df[results_df['threat_level'] == level]['threat_score']
    axes[2].hist(sub, bins=20, alpha=0.6, color=color_map[level], label=level, edgecolor='black')
axes[2].axvline(x=0.35, color='orange', linestyle='--', linewidth=2, label='Suspicious threshold')
axes[2].axvline(x=0.65, color='red',    linestyle='--', linewidth=2, label='Attack threshold')
axes[2].set_xlabel('Threat Score')
axes[2].set_ylabel('Number of IPs')
axes[2].set_title('Threat Score Distribution', fontweight='bold')
axes[2].legend()

plt.suptitle('Quantum-Inspired Threat Analysis — Training Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('07_threat_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Chart saved: 07_threat_analysis.png')

## 🧪 Step 9 — Testing on Test Dataset

In [ ]:
# ── Build test graph and run analysis ────────────────────────────────────────
print('🧪 Running full pipeline on TEST data...')
print('='*50)

# Build graph
test_graph_builder = NetworkGraphBuilder()
G_test = test_graph_builder.build(test_net)

# Quantum Walk
print('\n⚛️  Running Quantum Walk on test graph...')
test_qw   = QuantumWalkAnalyzer(steps=8)
test_qw_scores = test_qw.run(test_graph_builder.get_adjacency())
print(f'✅ Quantum Walk done: {len(test_qw_scores)} nodes scored')

# QFT
print('\n🌊 Running QFT on test data...')
test_qft  = QFTAnalyzer(bucket_count=16)
test_qft_scores = test_qft.run(test_net)
print(f'✅ QFT done: {len(test_qft_scores)} nodes scored')

# Threat Scoring
print('\n🎯 Scoring threats...')
test_scorer  = ThreatScorer()
test_threats = test_scorer.score(test_qw_scores, test_qft_scores)

test_summary = test_scorer.get_summary()
print(f'\n📊 Test Results Summary:')
for level in ['Attack', 'Suspicious', 'Normal']:
    count = test_summary.get(level, 0)
    emoji = '🔴' if level == 'Attack' else '🟡' if level == 'Suspicious' else '🟢'
    print(f'   {emoji} {level:12s}: {count} IPs')

In [ ]:
# ── Evaluate: compare predicted vs true labels ────────────────────────────────
print('\n📏 Evaluating predictions against true labels...')
print('='*50)

test_results_df = test_scorer.to_dataframe()

# Map per-IP prediction to per-row true label
# True label per IP = 'attack' if ANY connection from that IP is an attack
ip_true_label = {}
for _, row in test_net.iterrows():
    ip    = row['source_ip']
    label = row['true_label']
    if ip not in ip_true_label or label == 'attack':
        ip_true_label[ip] = label

# Align predicted vs true
y_true, y_pred = [], []
for _, row in test_results_df.iterrows():
    ip    = row['ip']
    pred  = 'attack' if row['threat_level'] in ('Attack', 'Suspicious') else 'normal'
    true  = ip_true_label.get(ip, 'normal')
    y_true.append(true)
    y_pred.append(pred)

# ── Metrics ───────────────────────────────────────────────────────────────────
acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, pos_label='attack', zero_division=0)
rec  = recall_score(y_true, y_pred, pos_label='attack', zero_division=0)
f1   = f1_score(y_true, y_pred, pos_label='attack', zero_division=0)

print(f'\n🏆 Performance Metrics:')
print(f'   Accuracy  : {acc:.4f}  ({acc*100:.1f}%)')
print(f'   Precision : {prec:.4f}')
print(f'   Recall    : {rec:.4f}')
print(f'   F1-Score  : {f1:.4f}')

print(f'\n📋 Full Classification Report:')
print(classification_report(y_true, y_pred, target_names=['attack','normal']))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred, labels=['attack','normal'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Predicted Attack','Predicted Normal'],
            yticklabels=['True Attack','True Normal'],
            ax=axes[0], linewidths=1)
axes[0].set_title('Confusion Matrix — Test Data', fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Metrics bar chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values  = [acc, prec, rec, f1]
bar_colors = ['#4488FF','#44BB44','#FFA500','#FF4444']
axes[1].bar(metrics, values, color=bar_colors, edgecolor='black', alpha=0.85)
for i, v in enumerate(values):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')
axes[1].set_ylim(0, 1.1)
axes[1].set_ylabel('Score')
axes[1].set_title('Model Performance Metrics', fontweight='bold')

plt.suptitle('Quantum-Inspired IDS — Test Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('08_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Chart saved: 08_evaluation.png')

## 📥 Step 10 — Download All Results

In [ ]:
# ── Save all results to CSV ───────────────────────────────────────────────────
print('💾 Saving results...')

# Train results
train_out = scorer.to_dataframe()
train_out.to_csv('train_results.csv', index=False)
print(f'✅ train_results.csv saved ({len(train_out)} rows)')

# Test results
test_out = test_scorer.to_dataframe()
test_out.to_csv('test_results.csv', index=False)
print(f'✅ test_results.csv saved ({len(test_out)} rows)')

# Summary report
report = {
    'metric':  ['Accuracy','Precision','Recall','F1-Score'],
    'value':   [round(acc,4), round(prec,4), round(rec,4), round(f1,4)]
}
pd.DataFrame(report).to_csv('model_report.csv', index=False)
print(f'✅ model_report.csv saved')

# Download all files
from google.colab import files
for fname in ['train_results.csv','test_results.csv','model_report.csv',
              '01_class_distribution.png','02_network_graph.png',
              '03_quantum_walk_scores.png','04_qft_periodicity.png',
              '06_qft_circuit_results.png','07_threat_analysis.png',
              '08_evaluation.png']:
    try:
        files.download(fname)
        print(f'⬇️  Downloaded: {fname}')
    except:
        print(f'⚠️  Could not download: {fname}')

print('\n✅ All done!')

---
## 📋 Summary

| Step | What happened |
|---|---|
| **Data Loading** | NSL-KDD Train + Test CSV loaded |
| **Feature Engineering** | IPs generated from attack types, timestamps bucketed |
| **Graph Builder** | IP communication graph (nodes=IPs, edges=traffic) |
| **Quantum Walk** | Amplitude spreads across graph → anomaly scores per IP |
| **QFT (Qiskit)** | DFT on time-series traffic → periodicity score per IP |
| **Threat Scoring** | `0.6×QW + 0.4×QFT` → Normal / Suspicious / Attack |
| **Evaluation** | Accuracy, Precision, Recall, F1 vs true labels |

### Algorithm Parameters
| Parameter | Value |
|---|---|
| Quantum Walk Steps | 8 |
| QFT Bucket Count | 16 |
| QW Weight | 0.6 |
| QFT Weight | 0.4 |
| Suspicious Threshold | 0.35 |
| Attack Threshold | 0.65 |